# Stage 1 Notebook: Grabbing Data From ZTF

1) create + activate an isolated env (conda shown; venv works to) and install the core stack. The .env will isolate where we load the data, before we store it in #ZTFDATA, which is where our image findings are going to be stored. Do this step in the terminal

## Cell 1 Configuration:
Make the project root importable so `import config` works from notebook/

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# IMPORTANT: import config FIRST — it loads .env and sets $ZTFDATA
# BEFORE ztfquery is imported (ztfquery reads $ZTFDATA at import time).
import configparser

#data-collection stack importing
from ztfquery import query
from ztfquery.io import LOCALSOURCE
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import matplotlib.pyplot as plt 

print("ZTFDATA resolves to :", LOCALSOURCE)
print("Setup OK - ztfquery will save downloads here.")

## Cell 2: Define the target field
For now, I'm going to be using a well-observed field. However, when collecting new data later on, I can always change the coordinates of where I want to look at later on

In [ ]:
TARGET_RA = 150.0
TARGET_DEC = 2.0
CUTOUT_SIZE = 0.01
FILTER = "zr"

print(f"Target field: RA={TARGET_RA}deg, Dec={TARGET_DEC} deg")
print(f"Search box:   {CUTOUT_SIZE}deg (~{CUTOUT_SIZE*3600:.0f} arcsec)")
print(f"Filter:       {FILTER}")


## Cell 3: Query ZTF Metadata (what images exist at our field)

In [ ]:
zquery = query.ZTFQuery()

zquery.load_metadata(
    radec=[TARGET_RA, TARGET_DEC],
    size=CUTOUT_SIZE,
    sql_query=f"fid=2",
)

meta = zquery.metatable
print("Rows returned (available images):", len(meta))
print("Columns:", list(meta.columns))
meta.head()

## Cell 4: Inspect and pick the best pixel grid
Narrowing all images to one clean pixel grid.

In [ ]:
#make sure all data is clean by making sure infobits == 0
clean = meta[meta["infobits"] == 0]
print(f"Clean frames (infobits==0): {len(clean)} of {len(meta)}")

# Count how many clean epochs each (field, ccdid, qid) grid has
grid_counts = (
    clean.groupby(["field", "ccdid", "qid"])
    .size()
    .sort_values(ascending=False)
)
print("\nTop pixel grids by clean-epoch count:")
print(grid_counts.head(10))

# Pick the winner: the grid with the most clean epochs
best_field, best_ccd, best_qid = grid_counts.index[0]
print(f"\nChosen grid -> field={best_field}, ccdid={best_ccd}, qid={best_qid}")

#Build our final sequence
sequence = (
    clean[
        (clean["field"] == best_field)
        & (clean["ccdid"] == best_ccd)
        & (clean["qid"] == best_qid)
    ]
    .sort_values("obsjd")
    .reset_index(drop=True)
)

print(f"\nFinal sequence: {len(sequence)} epochs on one pixel grid")
sequence[["obsjd", "field", "ccdid", "qid", "filtercode", "infobits"]].head()

### Cell 4B: Smaller sampling size
Easy to start small and scale from there

In [ ]:
N_EPOCHS = 5
subset = sequence.head(N_EPOCHS).reset_index(drop=True)

print(f"Selected {len(subset)} epochs to download (of {len(sequence)} available):")
subset[["obsjd", "field", "ccdid", "qid", "filtercode", "infobits"]]


## Cell 5: Download data and input in ztfdata folder

In [ ]:
chosen_indexes = meta.index[meta["obsjd"].isin(subset["obsjd"])].tolist()
print("Downloading metatable rows:", chosen_indexes)

zquery.download_data(
    "sciimg.fits",
    indexes=chosen_indexes,
    show_progress=True,
    nprocess=1,
)

print("Download call finished.")

## Cell 6: Verify the downloads: open one FITS, inspect header/WCS/data, view it.

In [ ]:
import glob
from astropy.wcs import WCS

# Find the sciimg files we downloaded (anywhere under ztfdata/sci/)
fits_files = sorted(glob.glob(str(Path(LOCALSOURCE) / "sci" / "**" / "*sciimg.fits"),
                              recursive=True))
print(f"Found {len(fits_files)} sciimg.fits files on disk:")
for f in fits_files:
    print("  ", Path(f).name)

# Open the FIRST one and look inside
path = fits_files[0]
print(f"\n--- Inspecting: {Path(path).name} ---")

with fits.open(path) as hdul:
    hdul.info()                       # structure: how many HDUs, their shapes
    header = hdul[0].header
    data   = hdul[0].data             # the 2D image as a NumPy array

print("\nImage shape (pixels):", data.shape)
print("Filter :", header.get("FILTER"))
print("Obs date:", header.get("OBSDATE", header.get("DATE-OBS")))
print("Exposure (s):", header.get("EXPTIME"))

# The WCS — the pixel<->sky map (the core of Stage 1)
wcs = WCS(header)
print("\nWCS is celestial:", wcs.has_celestial)
# Sanity check: what sky coordinate does the image CENTER map to?
ny, nx = data.shape
ra_c, dec_c = wcs.all_pix2world(nx/2, ny/2, 0)
print(f"Image center pixel -> sky: RA={ra_c:.4f} deg, Dec={dec_c:.4f} deg")
print(f"(Our target was RA={TARGET_RA}, Dec={TARGET_DEC} — should be close.)")

# View it (asinh-ish stretch so faint stars are visible, not just bright ones)
vmin, vmax = np.percentile(data[np.isfinite(data)], [5, 99])
plt.figure(figsize=(7, 7))
plt.imshow(data, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
plt.title(Path(path).name, fontsize=8)
plt.colorbar(label="pixel value")
plt.show()
